# K-Nearest Neighbors (KNN)

Implementing KNN, a non-parametric method used for classification. Since KNN is distance-based, we will include a scaling step.

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from metrics_utils import calculate_business_metrics


mlflow.set_experiment("Walmart-Sales-Classification")

<Experiment: artifact_location='file:///home/sarah/code/forth_year/data_science/project/walmart-sales-classification/src/models/mlruns/810253398722391848', creation_time=1777489467114, experiment_id='810253398722391848', last_update_time=1777489467114, lifecycle_stage='active', name='Walmart-Sales-Classification', tags={}>

In [2]:
train_df = pd.read_csv('../../data/model_ready/train.csv')
test_df = pd.read_csv('../../data/model_ready/test.csv')

features_selected = [
    "Size", "Store", "Dept", "CPI", "DeptFrequency", 
    "Week_cos", "IsPreHoliday", "Week_sin", "Fuel_Price", 
    "ConsumerConfRatio", "AvgMarkDownAmount"
]
target = "Sales_Class"
holiday_col = "IsHoliday"

X_train = train_df[features_selected]
y_train = train_df[target]
X_test = test_df[features_selected]
y_test = test_df[target]
test_holidays = test_df[holiday_col]

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
with mlflow.start_run(run_name="KNN"):
    model_path = 'knn_model.pkl'
    if os.path.exists(model_path):
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        print('Model loaded from pickle')
    else:
        params = {
            "n_neighbors": 5,
            "weights": "uniform",
            "metric": "minkowski"
        }
        model = KNeighborsClassifier(**params)
        model.fit(X_train_scaled, y_train)
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        print('Model trained and saved to pickle')

    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")

    biz_metrics = calculate_business_metrics(y_test, y_pred, test_holidays)

    mlflow.log_param("model_type", "KNN")
    mlflow.log_params(params)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_weighted", f1)
    mlflow.log_metric("holiday_accuracy", biz_metrics["holiday_accuracy"])
    mlflow.log_metric("weighted_classification_error", biz_metrics["weighted_classification_error"])

    mlflow.log_artifact(model_path)

    print(f"Accuracy: {acc:.4f}")
    print(f"Holiday Accuracy: {biz_metrics['holiday_accuracy']:.4f}")
    print(f"Weighted Error: {biz_metrics['weighted_classification_error']:.4f}")

Model trained and saved to pickle
Accuracy: 0.7463
Holiday Accuracy: 0.7220
Weighted Error: 0.2737
